# SolarTriage — ResNet18 pe GPU-ul Kaggle (tichetul 08)

Pași: vezi `ml/KAGGLE.md`. Necesită **Accelerator: GPU T4** și **Internet: On** (descarcă dataset-ul Raptor și greutățile ImageNet).
Intrare: dataset-ul Kaggle cu folderul `ml/` din repo (`train.py`, `data.py`, `export_onnx.py`, `splits/`).
Ieșiri în `/kaggle/working/`: `model.onnx`, `model_meta.json`, `best.pt`, `val_report.json`.

In [ ]:
EPOCHS = 20  # 1 pentru un v0 rapid

import importlib.util, shutil, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4"
print(torch.__version__, torch.cuda.get_device_name(0))

src = next(p.parent for p in Path("/kaggle/input").rglob("train.py") if (p.parent / "splits").is_dir())
REPO = Path("/tmp/repo")  # nu în /kaggle/working, ca imaginile să nu ajungă în output
shutil.copytree(src, REPO / "ml", dirs_exist_ok=True)
print("cod din", src)

def run(*args):
    subprocess.run([sys.executable, *args], cwd=REPO, check=True)

In [ ]:
# Imaginile Raptor (15,5 MB, SHA-256 verificat). Split-urile vin din repo, nu se regenerează.
run("-c", "import sys; sys.path.insert(0, 'ml'); import data; data.download(); data.extract()")

In [ ]:
run("ml/train.py", "--device", "cuda", "--epochs", str(EPOCHS), "--out", "/kaggle/working")

In [ ]:
if importlib.util.find_spec("onnx") is None:  # exportul ONNX are nevoie de pachetul onnx
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx"], check=True)
run("ml/export_onnx.py", "--ckpt", "/kaggle/working/best.pt", "--out-dir", "/kaggle/working")

In [ ]:
import json
r = json.load(open("/kaggle/working/val_report.json"))["best"]
print(f"epoca {r['epoch']}: val macro-F1 {r['macro_f1']:.4f}, accuracy {r['accuracy']:.4f}")
for c, f in r["f1_per_class"].items():
    print(f"  {c:<18} F1 {f:.3f}  (n={r['support'].get(c, 0)})")
!ls -la /kaggle/working